# Groundwater

In [2]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Load data

In [93]:
!pip install -q xgboost

import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [ ]:
gw_df = pd.read_csv("groundwater_model_dataset.csv")

print("Dataset shape:", gw_df.shape)

print("\nColumns:")
print(gw_df.columns.tolist())

Dataset shape: (104090, 23)

Columns:
['Data Acquisition Time', 'District LGD Code', 'District', 'rainfall', 'groundwater_level', 'year', 'month', 'day', 'hour', 'day_of_week', 'season', 'rainfall_lag_1', 'rainfall_lag_3', 'rainfall_lag_6', 'groundwater_lag_1', 'groundwater_lag_3', 'groundwater_lag_6', 'rainfall_roll_3', 'rainfall_roll_6', 'groundwater_roll_3', 'groundwater_roll_6', 'rainfall_diff_1', 'groundwater_diff_1']


In [95]:
X = gw_df.drop(columns=["groundwater_level", "Data Acquisition Time"])
y = gw_df["groundwater_level"]

In [96]:
leak_cols = [
    "groundwater_roll_3", "groundwater_roll_6",
    "groundwater_lag_1", "groundwater_lag_3",
    "groundwater_lag_6"
]

X = X.drop(columns=leak_cols)

In [97]:
categorical_cols = ["District", "season"]
numerical_cols = [col for col in X.columns if col not in categorical_cols]

print("\nCategorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)



Categorical Columns: ['District', 'season']
Numerical Columns: ['District LGD Code', 'rainfall', 'year', 'month', 'day', 'hour', 'day_of_week', 'rainfall_lag_1', 'rainfall_lag_3', 'rainfall_lag_6', 'rainfall_roll_3', 'rainfall_roll_6', 'rainfall_diff_1', 'groundwater_diff_1']


# Preprocessor

In [98]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

le = LabelEncoder()
y_train = le.fit_transform(y_train)

# Train data split

In [99]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (83272, 16)
Test shape: (20818, 16)


# Modelling

In [100]:
models = {
    "Linear Regression": LinearRegression(),

    "Random Forest (CPU fast)": RandomForestRegressor(
        n_estimators=100,
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost (T4 GPU)": XGBRegressor(
    tree_method="hist",
    device="cuda",
    n_estimators=300,
    max_depth=6,
    predictor="gpu_predictor",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
}

In [ ]:
!nvidia-smi

Sun Apr  5 10:24:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             27W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [101]:
corr = df.corr(numeric_only=True)

print("\nCorrelation with flood_risk_score:")
print(corr["flood_risk_score"].sort_values(ascending=False))


Correlation with flood_risk_score:
flood_risk_score      1.000000
river_water_level     0.903284
river_roll_3          0.899019
river_lag_1           0.894575
river_roll_6          0.893620
river_lag_3           0.883525
year                  0.649152
groundwater_level     0.476478
groundwater_roll_3    0.473809
groundwater_roll_6    0.470003
groundwater_lag_1     0.469284
groundwater_lag_3     0.460428
District LGD Code     0.229947
river_diff_1          0.061557
rainfall_diff_1       0.053500
hour                  0.052875
day_of_week           0.046026
rainfall              0.038507
groundwater_diff_1    0.037802
day                  -0.008673
rainfall_roll_3      -0.008871
rainfall_lag_1       -0.020358
rainfall_roll_6      -0.043744
rainfall_lag_3       -0.044291
month                -0.227775
Name: flood_risk_score, dtype: float64


In [103]:
results = []
for name, model in models.items():
  print(f"\n===== {name} =====")
  pipeline = Pipeline([
    ("preprocessor", preprocessor), ("model", model)
  ])
  start = time.time()
  pipeline.fit(X_train, y_train)
  train_time = time.time() - start
  y_pred = pipeline.predict(X_test)
  mae = mean_absolute_error(y_test, y_pred)
  rmse = np.sqrt(mean_squared_error(y_test, y_pred))
  r2 = r2_score(y_test, y_pred)
  results.append([name, mae, rmse, r2, train_time])
  print(f"Training Time: {train_time:.2f}) sec")
  print("MAE:", mae)
  print("R²:", r2)
  print("RMSE:", rmse)


===== Linear Regression =====
Training Time: 1.07) sec
MAE: 7.174406338247941
R²: 0.3229014529905686
RMSE: 26.59248087295854

===== Random Forest (CPU fast) =====
Training Time: 1094.97) sec
MAE: 0.6757480330370618
R²: 0.965072368458264
RMSE: 6.039729425988344

===== XGBoost (T4 GPU) =====


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:09:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training Time: 0.87) sec
MAE: 2.364097825048385
R²: 0.9498167933054464
RMSE: 7.239559123160559


In [ ]:
import seaborn as sns

corr = gw_df.corr(numeric_only=True)
print(corr["groundwater_level"].sort_values(ascending=False))

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2 Score", "Training Time (s)"]
)

results_df = results_df.sort_values("R2 Score", ascending=False)

print("\n===== FINAL REGRESSION RESULTS =====")
print(results_df)

results_df.to_csv("groundwater_regression_results.csv", index=False)
print("\nSaved results as groundwater_regression_results.csv")


===== FINAL REGRESSION RESULTS =====
                      Model       MAE       RMSE  R2 Score  Training Time (s)
1  Random Forest (CPU fast)  0.675748   6.039729  0.965072        1078.438373
2          XGBoost (T4 GPU)  2.364098   7.239559  0.949817           1.009269
0         Linear Regression  7.174406  26.592481  0.322901           0.258108

Saved results as groundwater_regression_results.csv


# Flood

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from lightgbm import LGBMClassifier


In [18]:
df = pd.read_csv("flood_model_dataset.csv")

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Dataset shape: (31433, 29)

Columns: ['Data Acquisition Time', 'District LGD Code', 'District', 'rainfall', 'river_water_level', 'groundwater_level', 'year', 'month', 'day', 'hour', 'day_of_week', 'season', 'rainfall_lag_1', 'rainfall_lag_3', 'river_lag_1', 'river_lag_3', 'groundwater_lag_1', 'groundwater_lag_3', 'rainfall_roll_3', 'rainfall_roll_6', 'river_roll_3', 'river_roll_6', 'groundwater_roll_3', 'groundwater_roll_6', 'rainfall_diff_1', 'river_diff_1', 'groundwater_diff_1', 'flood_risk_score', 'flood_risk_category']


In [19]:
features=df.columns.tolist()
features.remove('flood_risk_category')
features.remove('flood_risk_score')
features.remove('Data Acquisition Time')
target='flood_risk_category'

In [20]:
train = df[df['Data Acquisition Time'] < '2025-04-01']
test = df[df['Data Acquisition Time'] >= '2025-04-01']
X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]
print(f"Train Shape: {X_train.shape}\nTest Shape: {X_test.shape}")

Train Shape: (25577, 26)
Test Shape: (5856, 26)


In [21]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)  # e.g., Low=0, Medium=1, High=2
y_test = le.fit_transform(y_test)
print("\nEncoded target classes:", le.classes_)


Encoded target classes: ['High' 'Low' 'Medium']


In [22]:
'''leak_cols = [
    "river_roll_3", "river_roll_6",
    "river_lag_1", "river_lag_3",
    "groundwater_roll_3", "groundwater_roll_6",
    "groundwater_lag_1", "groundwater_lag_3"
]
X = X.drop(columns=leak_cols)'''

# Identify categorical and numerical columns
categorical_cols = ["District", "season"]
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]


In [23]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

In [24]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Random Forest (CPU fast)": RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost (GPU)": XGBClassifier(
        tree_method="hist",
        predictor="gpu_predictor",
        device="cuda",
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric="mlogloss"
    ),
    "LightGBM (GPU)": LGBMClassifier(
        device="gpu",                 
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,                
        subsample=0.8,                
        colsample_bytree=0.8,         
        min_child_samples=20,         
        reg_alpha=0.1,                
        reg_lambda=0.1,               
        random_state=42
    )
}


In [25]:
results = []

for name, model in models.items():
    print(f"\n===== {name} =====")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    start = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    results.append([name, acc, prec, rec, f1, train_time])

    print(f"Training Time: {train_time:.2f} sec")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 Score :", f1)



===== Logistic Regression =====
Training Time: 0.47 sec
Accuracy : 0.9769467213114754
Precision: 0.9772933687511887
Recall   : 0.9769467213114754
F1 Score : 0.9770297425357901

===== Random Forest (CPU fast) =====
Training Time: 5.76 sec
Accuracy : 0.9863387978142076
Precision: 0.9864505527594584
Recall   : 0.9863387978142076
F1 Score : 0.9863508864261011

===== XGBoost (GPU) =====


e:\Vignesh\VIT\Sem 6\Predictive\Project\Flood-Risk-Assessment-and-Groundwater-Level-Prediction-System\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:37:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
e:\Vignesh\VIT\Sem 6\Predictive\Project\Flood-Risk-Assessment-and-Groundwater-Level-Prediction-System\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:37:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
e:\Vignesh\VIT\Sem 6\Predictive\Project\Flood-Risk-Assessment-and-Groundwater-Level-Prediction-System\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:37:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor", "use_label_encoder

Training Time: 4.29 sec
Accuracy : 0.9948770491803278
Precision: 0.9948826713313098
Recall   : 0.9948770491803278
F1 Score : 0.994877676466996

===== LightGBM (GPU) =====
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 4710
[LightGBM] [Info] Number of data points in the train set: 25577, number of used features: 44
[LightGBM] [Info] Using GPU Device: Intel(R) Iris(R) Xe Graphics, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 24 dense feature groups (0.59 MB) transferred to GPU in 0.004414 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score -1.121230
[LightGBM] [Info] Start training from score -1.074241
[LightGBM] [Info] Start training from score -1.100922
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

e:\Vignesh\VIT\Sem 6\Predictive\Project\Flood-Risk-Assessment-and-Groundwater-Level-Prediction-System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [26]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "Training Time (s)"]
).sort_values("F1 Score", ascending=False)

print("\n===== FINAL CLASSIFICATION RESULTS =====")
print(results_df)

# Save results
results_df.to_csv("flood_classification_results.csv", index=False)
print("\nSaved results as flood_classification_results.csv")


===== FINAL CLASSIFICATION RESULTS =====
                      Model  Accuracy  Precision    Recall  F1 Score  \
2             XGBoost (GPU)  0.994877   0.994883  0.994877  0.994878   
3            LightGBM (GPU)  0.994023   0.994028  0.994023  0.994024   
1  Random Forest (CPU fast)  0.986339   0.986451  0.986339  0.986351   
0       Logistic Regression  0.976947   0.977293  0.976947  0.977030   

   Training Time (s)  
2           4.287560  
3          43.309500  
1           5.758033  
0           0.474376  

Saved results as flood_classification_results.csv
